In [1]:
from time import perf_counter
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from aeon.transformations.collection.convolution_based import MultiRocket
from tscglue.data_loader import load_fold
from tscglue.models_tsfm import RidgeClassifierCVDecisionProba

dataset = "Wine"
fold = 0
X_train, y_train, X_test, y_test = load_fold(dataset, fold)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, classes: {np.unique(y_train)}")

2026-03-18 16:28:24.455389: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


X_train: (57, 1, 234), X_test: (54, 1, 234), classes: ['1' '2']


In [2]:
# Extract MultiRocket features
mr = MultiRocket(random_state=42)

t0 = perf_counter()
Xt_train = mr.fit_transform(X_train)
Xt_test = mr.transform(X_test)
t_mr = perf_counter() - t0

print(f"MultiRocket features: {Xt_train.shape[1]}")
print(f"Transform time: {t_mr:.1f}s")

MultiRocket features: 49728
Transform time: 1.9s


In [3]:
# 1) Ridge on MultiRocket features (with scaling)
ridge_pipe = make_pipeline(
    StandardScaler(),
    RidgeClassifierCVDecisionProba(alphas=np.logspace(-3, 3, 10)),
)

t0 = perf_counter()
ridge_pipe.fit(Xt_train, y_train)
y_pred_ridge = ridge_pipe.predict(Xt_test)
t_ridge = perf_counter() - t0

acc_ridge = accuracy_score(y_test, y_pred_ridge)
print(f"Ridge on MultiRocket: acc={acc_ridge:.4f} ({t_ridge:.1f}s)")

Ridge on MultiRocket: acc=0.8333 (0.2s)


In [ ]:
# 2) TabICL on PCA-reduced MultiRocket features
from tabicl import TabICLClassifier

tabicl_pipe = make_pipeline(
    StandardScaler(),
    # PCA(n_components=0.99),
    TabICLClassifier(device="cpu", n_jobs=-1),
)

t0 = perf_counter()
tabicl_pipe.fit(Xt_train, y_train)
y_pred_tabicl = tabicl_pipe.predict(Xt_test)
t_tabicl = perf_counter() - t0

acc_tabicl = accuracy_score(y_test, y_pred_tabicl)
print(f"PCA components: {tabicl_pipe[1].n_components_}")
print(f"TabICL on PCA-reduced MultiRocket: acc={acc_tabicl:.4f} ({t_tabicl:.1f}s)")

In [ ]:
# Summary
print(f"{'Method':<45} {'Accuracy':>10} {'Time':>8}")
print("-" * 65)
print(f"{'Ridge on MultiRocket (scaled)':<45} {acc_ridge:>10.4f} {t_ridge:>7.1f}s")
print(f"{'TabICL on MultiRocket (PCA 99%)':<45} {acc_tabicl:>10.4f} {t_tabicl:>7.1f}s")